## Paso 1 — Cargamos el dataset ya tipado

Mismo parquet que los notebooks anteriores. Este notebook cubre las
Fases 11 y 12 del plan (§12-13): consolidación de limitaciones y
síntesis final de todo el EDA.


In [1]:
import pandas as pd
from pathlib import Path

INTERIM_DIR = Path("../datos/interim")
df = pd.read_parquet(INTERIM_DIR / "viajes_btura_tipado.parquet")

print(df.shape)


(41237, 17)


In [2]:
# H-001: FO no es clave única
print("FO valores repetidos:", df["fo"].duplicated().sum())
print("FO únicos:", df["fo"].nunique(), "de", df.shape[0], "filas")


FO valores repetidos: 130
FO únicos: 41107 de 41237 filas


In [3]:
# H-002: Tarifa Venta al Cliente = Flete Producción FO (72,2%)
validos = df[(df["tarifaVentaAlCliente"] > 0) & (df["fleteProduccionFo"] > 0)]
pct_igual = (validos["tarifaVentaAlCliente"] == validos["fleteProduccionFo"]).mean() * 100
print(f"% de filas con Tarifa = Flete: {pct_igual:.2f}%")


% de filas con Tarifa = Flete: 72.23%


In [ ]:
# H-003: salto de flete mediano en marzo 2026 (reproducible aquí; la causa SICE-TAC/salario mínimo es externa)
flete_valido = df[df["fleteProduccionFo"] > 0].copy()
flete_valido["mes"] = flete_valido["fechaCreacionFo"].dt.to_period("M").astype(str)
flete_por_mes = flete_valido.groupby("mes")["fleteProduccionFo"].median()
print((flete_por_mes.pct_change() * 100).round(1))


In [ ]:
# H-005: flete más alto (la comparación contra tblSAPTarifas es externa)
print(df.nlargest(5, "fleteProduccionFo")[["fo", "fleteProduccionFo", "distanciaTotal", "origen", "destino"]])


In [ ]:
# H-006: viajes de TC_CAMNETA con flete exacto de $800.000 (el porqué es externo)
camioneta = df[(df["idMedioTransporteFo"] == "TC_CAMNETA") & (df["fleteProduccionFo"] == 800000)]
print(camioneta[["fo", "fleteProduccionFo", "distanciaTotal", "origen", "destino"]])


In [4]:
# H-007: Vehículo siempre tiene el mismo Tipo
tipos_por_vehiculo = df.groupby("vehiculo")["tipo"].nunique()
print(f"Vehículos con más de un Tipo: {(tipos_por_vehiculo > 1).sum()} de {tipos_por_vehiculo.shape[0]}")


Vehículos con más de un Tipo: 0 de 3418


In [ ]:
# H-009: Origen = Destino, y el residual sospechoso
od_iguales = df[df["origen"] == df["destino"]]
print(f"Origen=Destino: {od_iguales.shape[0]} ({od_iguales.shape[0]/df.shape[0]*100:.2f}%)")

residual = od_iguales[od_iguales["idClaseDocumentoFwo"].isin(["OPIM", "OPTN"])]
print(f"Residual (OPIM+OPTN): {residual.shape[0]}")
print(f"De esos, con Distancia Total = 10: {(residual['distanciaTotal'] == 10).sum()}")


In [6]:
# H-010: Concentración de clientes (HHI)
conteo_cliente = df["bpCliente"].value_counts()
participacion = conteo_cliente / conteo_cliente.sum()
hhi = (participacion ** 2).sum() * 10000
print(f"HHI: {hhi:.0f}")
print(f"Top 10: {(conteo_cliente.head(10).sum() / conteo_cliente.sum() * 100):.1f}%")


HHI: 331
Top 10: 39.7%


In [7]:
# H-011: Distancia Total idéntica dentro del mismo par origen-destino (ejemplos)
for origen, destino in [("BARBOSA", "BUENAVENTURA"), ("BELLO", "BUENAVENTURA")]:
    subset = df[(df["origen"] == origen) & (df["destino"] == destino)]
    print(f"{origen}->{destino}: n={subset.shape[0]}, std distancia={subset['distanciaTotal'].std()}")


BARBOSA->BUENAVENTURA: n=127, std distancia=0.0
BELLO->BUENAVENTURA: n=301, std distancia=0.0


## Cierre del EDA

El detalle completo de hallazgos, limitaciones, log de supuestos y
recomendaciones vive en `reportes/hallazgos.md` y
`reportes/limitaciones.md`. Las celdas de arriba reproducen los números
verificables desde este parquet; los hallazgos marcados 🔎 en esos
archivos se verificaron contra la base de datos operativa de la empresa
y no son reproducibles desde aquí.
